<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/bioassay/bioassay_lean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bioassay: Bayesian workflow

**Short Bayesian course — worked example**

$$
\text{data}
\rightarrow
\text{model}
\rightarrow
\text{prior predictive}
\rightarrow
\text{fit}
\rightarrow
LD50
\rightarrow
\text{posterior predictive}
$$

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260923
azp.style.use("arviz-variat")

# Figures used in the slides: Slides/figures/<notebook>_<section>[_<n>].svg
NOTEBOOK = "bioassay_lean"
FIG_DIR = Path("../Slides/figures") if Path("../Slides").is_dir() else Path("figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_slide_figure(fig, section, number=None):
    """Save a Matplotlib figure or ArviZ PlotCollection for the slides."""
    suffix = f"_{number}" if number else ""
    fig.savefig(
        FIG_DIR / f"{NOTEBOOK}_{section}{suffix}.svg", bbox_inches="tight", transparent=True
    )

print("PyMC:", pm.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## 1. Data

In [ ]:
dose = np.array([-0.86, -0.30, -0.05, 0.73])
n = np.array([5, 5, 5, 5])
deaths = np.array([0, 1, 3, 5])

bioassay = pd.DataFrame(
    {
        "dose_log_g_ml": dose,
        "animals": n,
        "deaths": deaths,
    }
)
bioassay["proportion_dead"] = bioassay["deaths"] / bioassay["animals"]
bioassay

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(dose, deaths / n, s=70)
ax.set(
    xlabel="Dose log(g/ml)",
    ylabel="Observed proportion dead",
    ylim=(-0.05, 1.05),
)
plt.show()

## 2. Model

$$
y_j\sim\operatorname{Binomial}(n_j,p_j),
\qquad
\operatorname{logit}(p_j)=\alpha+\beta x_j
$$

$$
\alpha\sim N(0,5),
\qquad
\beta\sim\operatorname{HalfNormal}(5).
$$

In [ ]:
coords = {"dose_log_g_ml": dose}

with pm.Model(coords=coords) as model:
    dose_data = pm.Data("dose", dose, dims="dose_log_g_ml")
    n_data = pm.Data("n", n, dims="dose_log_g_ml")

    alpha = pm.Normal("alpha", mu=0, sigma=5)
    beta = pm.HalfNormal("beta", sigma=5)

    logit_p = alpha + beta * dose_data

    # Retain p because the later dose-response plots use the probability scale directly.
    p = pm.Deterministic(
        "p",
        pm.math.sigmoid(logit_p),
        dims="dose_log_g_ml",
    )

    ld50_log_g_ml = pm.Deterministic(
        "LD50_log_g_ml",
        -alpha / beta,
    )
    ld50_mg_ml = pm.Deterministic(
        "LD50_mg_ml",
        1000 * pm.math.exp(ld50_log_g_ml),
    )

    pm.Binomial(
        "deaths",
        n=n_data,
        logit_p=logit_p,
        observed=deaths,
        dims="dose_log_g_ml",
    )

## 3. Prior predictive

**Question:** What kinds of observed death counts do these priors make plausible?

In [ ]:
with model:
    prior = pm.sample_prior_predictive(
        draws=1000,
        var_names=["alpha", "beta", "deaths"],
        random_seed=RANDOM_SEED,
    )

In [ ]:
azp.plot_dist(
    prior,
    var_names=["deaths"],
    group="prior_predictive",
    sample_dims=["chain", "draw"],
    kind="hist",
    cols=["dose_log_g_ml"],
    visuals={
        "credible_interval": False,
        "point_estimate": False,
        "point_estimate_text": False,
    },
)

plt.gcf().supxlabel("Deaths out of 5")
plt.gcf().suptitle("Prior predictive deaths by dose (log(g/ml))")

Each panel is the prior predictive distribution of the number of deaths at one dose.

The same draws seen as whole experiments: each line is one dataset the priors say we could have seen.

In [ ]:
simulated = prior["prior_predictive"]["deaths"].isel(chain=0, draw=slice(0, 30)) / n

fig, ax = plt.subplots(figsize=(4.8, 3.4))
lines = ax.plot(dose, simulated.values.T, color="C0", alpha=0.35)
lines[0].set_label("simulated experiments")
ax.plot(dose, deaths / n, "o-", color="black", lw=2.5, ms=8, label="observed")
ax.set(xlabel="Dose log(g/ml)", ylabel="Proportion dead", ylim=(-0.05, 1.05))
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1), ncols=2, frameon=False)
save_slide_figure(fig, "prior-predictive")
plt.show()

## 4. Fit and diagnose

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        nuts={"target_accept": 0.90},
        random_seed=RANDOM_SEED,
    )

In [ ]:
print("Divergences:", idata["sample_stats"]["diverging"].sum().item())

azs.summary(
    idata,
    var_names=["alpha", "beta"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(
    idata,
    var_names=["alpha", "beta"],
)

## 5. Posterior dose-response fit

Calculate expected deaths only now, when we need them for the regression plot.

In [ ]:
idata["posterior"]["expected_deaths"] = (
    idata["posterior"]["p"] * idata["constant_data"]["n"]
)

azp.plot_lm(
    idata,
    x="dose",
    y="expected_deaths",
    y_obs="deaths",
    group="posterior",
    plot_dim="dose_log_g_ml",
    ci_prob=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
    smooth=False,
    visuals={"observed_scatter": False},
)

plt.scatter(dose, deaths, zorder=3)
plt.xlabel("Dose log(g/ml)")
plt.ylabel("Deaths out of 5")

## 6. LD50

$$
LD50=-\frac{\alpha}{\beta}.
$$

It is stored as a PyMC deterministic quantity.

In [ ]:
azs.summary(
    idata,
    var_names=["LD50_log_g_ml", "LD50_mg_ml"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_dist(
    idata,
    var_names=["LD50_mg_ml"],
    point_estimate="median",
    ci_prob=0.90,
    ci_kind="hdi",
)

## 7. Posterior predictive

**Question:** Can the fitted model generate death counts like those observed?

In [ ]:
with model:
    pm.sample_posterior_predictive(
        idata,
        var_names=["deaths"],
        extend_inferencedata=True,
        random_seed=RANDOM_SEED,
    )

In [ ]:
azp.plot_ppc_interval(
    idata,
    var_names=["deaths"],
    ci_probs=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
)

plt.xticks(np.arange(len(dose)), [f"{value:g}" for value in dose])
plt.xlabel("Dose log(g/ml)")

## 8. Dose-response prediction

Predict the expected number of deaths on a dense dose grid. This shows uncertainty in the dose-response curve; the posterior predictive check above also includes binomial outcome variability.

In [ ]:
dose_grid = np.linspace(-1.0, 1.0, 101)

with model:
    pm.set_data(
        {
            "dose": dose_grid,
            "n": np.full(dose_grid.size, 5),
        },
        coords={"dose_log_g_ml": dose_grid},
    )

    pm.sample_posterior_predictive(
        idata,
        var_names=["p"],
        predictions=True,
        extend_inferencedata=True,
        random_seed=RANDOM_SEED,
    )

    # Restore the observed design after prediction.
    pm.set_data(
        {
            "dose": dose,
            "n": n,
        },
        coords={"dose_log_g_ml": dose},
    )

idata["predictions"]["expected_deaths"] = 5 * idata["predictions"]["p"]

In [ ]:
azp.plot_lm(
    idata,
    x="dose",
    y="expected_deaths",
    y_obs="deaths",
    group="predictions",
    plot_dim="dose_log_g_ml",
    ci_prob=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
    smooth=False,
    visuals={"observed_scatter": False},
)

plt.scatter(dose, deaths, zorder=3)
plt.xlabel("Dose log(g/ml)")
plt.ylabel("Deaths out of 5")

## 9. Explore

- Remove the positive-slope constraint.
- Change the priors and rerun the prior predictive check.
- Predict a new group at one chosen dose.

Source: Gelman & Vehtari, *Bayesian Workflow*, §3.5.